# SVM vs Random Forest - CICIDS2017 (WGAN-GP Balanced - GPU)
Benchmark on cleaned CICIDS2017 dataset with WGAN-GP data augmentation and 70-30 train/test split

**Strategy:**
- Majority classes (>10,286 samples): Undersample to 50,000
- Minority classes (≤10,286 samples): WGAN-GP augment to 20,000
- WGAN-GP runs on GPU for faster training

In [ ]:
!pip install imbalanced-learn -q

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! WGAN-GP will run on CPU (much slower).")
    print("Go to Runtime -> Change runtime type -> T4 GPU")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

USE_GPU = torch.cuda.is_available()
DEVICE = torch.device('cuda' if USE_GPU else 'cpu')
print(f"Libraries imported successfully!")
print(f"WGAN-GP will use: {'GPU ✓' if USE_GPU else 'CPU ✗'}")

## Step 1: Mount Drive & Load Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/data_processed/cicids2017_cleaned.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nOriginal Class Distribution:")
print(df['Label'].value_counts())

## Step 2: Train/Test Split (70-30)

In [ ]:
X = df.drop('Label', axis=1)
y = df['Label']
feature_columns = X.columns.tolist()

print(f"Total features: {X.shape[1]}")
print(f"Total samples: {X.shape[0]}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(f"\nTrain set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nTraining set class distribution (BEFORE balancing):")
print(y_train.value_counts())

## Step 3: Undersample Majority Classes to 50,000

In [ ]:
threshold = 10286
target_minority = 20000

under_strategy = {}
for label, count in Counter(y_train).items():
    if count > threshold:
        under_strategy[label] = 50000

print(f"Undersampling classes {list(under_strategy.keys())} -> 50,000 each")

under_sampler = RandomUnderSampler(sampling_strategy=under_strategy, random_state=42)
X_train_under, y_train_under = under_sampler.fit_resample(X_train, y_train)

print(f"After undersampling: {len(y_train_under):,} samples")
print(pd.Series(y_train_under).value_counts())

## Step 4: Define WGAN-GP Model Architecture

In [ ]:
class Generator(nn.Module):
    """Generator network for WGAN-GP."""
    def __init__(self, noise_dim, output_dim):
        super(Generator, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim, 256),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(512),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.BatchNorm1d(256),
            nn.Linear(256, output_dim),
        )

    def forward(self, z):
        return self.net(z)


class Critic(nn.Module):
    """Critic (Discriminator) network for WGAN-GP."""
    def __init__(self, input_dim):
        super(Critic, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        return self.net(x)


def compute_gradient_penalty(critic, real_data, fake_data, device):
    """Compute gradient penalty for WGAN-GP."""
    batch_size = real_data.size(0)
    alpha = torch.rand(batch_size, 1, device=device)
    alpha = alpha.expand_as(real_data)

    interpolated = (alpha * real_data + (1 - alpha) * fake_data).requires_grad_(True)
    critic_interpolated = critic(interpolated)

    gradients = torch.autograd.grad(
        outputs=critic_interpolated,
        inputs=interpolated,
        grad_outputs=torch.ones_like(critic_interpolated),
        create_graph=True,
        retain_graph=True,
    )[0]

    gradients = gradients.view(batch_size, -1)
    gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gradient_penalty


print("WGAN-GP architecture defined ✓")

## Step 5: WGAN-GP Training Function

In [ ]:
def train_wgan_gp(real_data_np, n_samples, device,
                   noise_dim=128, epochs=300, batch_size=64,
                   critic_iters=5, lambda_gp=10,
                   lr=1e-4, verbose=True):
    """
    Train a WGAN-GP on the given real data and generate n_samples synthetic samples.

    Args:
        real_data_np: numpy array of real samples (n_real, n_features)
        n_samples: number of synthetic samples to generate
        device: torch device (cuda or cpu)
        noise_dim: dimension of the noise vector for the generator
        epochs: number of training epochs
        batch_size: training batch size
        critic_iters: number of critic updates per generator update
        lambda_gp: gradient penalty coefficient
        lr: learning rate for both generator and critic
        verbose: whether to print progress

    Returns:
        synthetic_data: numpy array of generated samples (n_samples, n_features)
    """
    n_features = real_data_np.shape[1]
    n_real = real_data_np.shape[0]

    # Normalize real data to help GAN training
    data_scaler = MinMaxScaler(feature_range=(-1, 1))
    real_data_scaled = data_scaler.fit_transform(real_data_np)

    # Create DataLoader
    real_tensor = torch.FloatTensor(real_data_scaled).to(device)
    dataset = TensorDataset(real_tensor)

    # Ensure batch_size doesn't exceed dataset size
    effective_batch_size = min(batch_size, n_real)
    if effective_batch_size < 2:
        effective_batch_size = 2

    dataloader = DataLoader(dataset, batch_size=effective_batch_size, shuffle=True, drop_last=True)

    # Initialize models
    generator = Generator(noise_dim, n_features).to(device)
    critic = Critic(n_features).to(device)

    # Optimizers (Adam with betas as recommended in WGAN-GP paper)
    opt_gen = optim.Adam(generator.parameters(), lr=lr, betas=(0.0, 0.9))
    opt_critic = optim.Adam(critic.parameters(), lr=lr, betas=(0.0, 0.9))

    # Training loop
    for epoch in range(epochs):
        epoch_critic_loss = 0.0
        epoch_gen_loss = 0.0
        n_critic_batches = 0
        n_gen_batches = 0

        for batch_idx, (real_batch,) in enumerate(dataloader):
            cur_batch_size = real_batch.size(0)

            # ====== Train Critic ======
            for _ in range(critic_iters):
                noise = torch.randn(cur_batch_size, noise_dim, device=device)
                fake_data = generator(noise).detach()

                critic_real = critic(real_batch).mean()
                critic_fake = critic(fake_data).mean()
                gp = compute_gradient_penalty(critic, real_batch, fake_data, device)

                critic_loss = critic_fake - critic_real + lambda_gp * gp

                opt_critic.zero_grad()
                critic_loss.backward()
                opt_critic.step()

                epoch_critic_loss += critic_loss.item()
                n_critic_batches += 1

            # ====== Train Generator ======
            noise = torch.randn(cur_batch_size, noise_dim, device=device)
            fake_data = generator(noise)
            gen_loss = -critic(fake_data).mean()

            opt_gen.zero_grad()
            gen_loss.backward()
            opt_gen.step()

            epoch_gen_loss += gen_loss.item()
            n_gen_batches += 1

        # Print progress every 50 epochs
        if verbose and (epoch + 1) % 50 == 0:
            avg_c = epoch_critic_loss / max(n_critic_batches, 1)
            avg_g = epoch_gen_loss / max(n_gen_batches, 1)
            print(f"  Epoch [{epoch+1}/{epochs}] | Critic Loss: {avg_c:.4f} | Gen Loss: {avg_g:.4f}")

    # ====== Generate Synthetic Samples ======
    generator.eval()
    synthetic_list = []
    remaining = n_samples

    with torch.no_grad():
        while remaining > 0:
            gen_batch_size = min(remaining, 10000)
            noise = torch.randn(gen_batch_size, noise_dim, device=device)
            fake = generator(noise).cpu().numpy()
            synthetic_list.append(fake)
            remaining -= gen_batch_size

    synthetic_scaled = np.concatenate(synthetic_list, axis=0)

    # Inverse-transform back to original scale
    synthetic_data = data_scaler.inverse_transform(synthetic_scaled)

    return synthetic_data


print("WGAN-GP training function defined ✓")

## Step 6: WGAN-GP Augmentation on GPU (Minority Classes → 20,000)

In [ ]:
class_counts = Counter(y_train_under)
minority_classes = {label: count for label, count in class_counts.items() if count <= threshold}

print("=" * 60)
print(f"WGAN-GP DATA AUGMENTATION ({'GPU' if USE_GPU else 'CPU'})")
print("=" * 60)
print(f"\nTarget for minority classes: {target_minority}")
print(f"\nClasses to augment:")
for label, count in sorted(minority_classes.items()):
    print(f"  Label {label}: {count:,} -> {target_minority:,} (+{target_minority - count:,} synthetic)")

In [ ]:
train_df = pd.DataFrame(X_train_under, columns=feature_columns)
train_df['Label'] = y_train_under.values

synthetic_dfs = []

for label, current_count in sorted(minority_classes.items()):
    samples_needed = target_minority - current_count
    if samples_needed <= 0:
        print(f"Label {label}: Already has {current_count} samples, skipping.")
        continue

    print(f"\n{'='*50}")
    print(f"WGAN-GP for Label {label}: {current_count} -> {target_minority} (+{samples_needed} synthetic)")
    print(f"{'='*50}")

    class_data = train_df[train_df['Label'] == label].drop('Label', axis=1)

    # Skip if too few samples to train WGAN-GP reliably
    if len(class_data) < 5:
        print(f"  ⚠️ Label {label}: Only {len(class_data)} samples — too few for WGAN-GP.")
        print(f"  Duplicating existing samples to reach target instead.")
        duplicated = class_data.sample(n=samples_needed, replace=True, random_state=42)
        # Add small noise to avoid exact duplicates
        noise = np.random.normal(0, 0.01, duplicated.shape)
        duplicated = duplicated + noise
        duplicated['Label'] = label
        synthetic_dfs.append(duplicated)
        print(f"  Label {label}: Created {len(duplicated)} noisy duplicates ✓")
        continue

    # Adjust hyperparameters based on class size
    if current_count < 50:
        epochs = 500
        batch_size = max(2, current_count // 2 * 2)
    elif current_count < 500:
        epochs = 400
        batch_size = max(2, min(64, current_count) // 2 * 2)
    else:
        epochs = 300
        batch_size = max(2, min(256, current_count) // 2 * 2)

    try:
        synthetic_data = train_wgan_gp(
            real_data_np=class_data.values,
            n_samples=samples_needed,
            device=DEVICE,
            noise_dim=128,
            epochs=epochs,
            batch_size=batch_size,
            critic_iters=5,
            lambda_gp=10,
            lr=1e-4,
            verbose=True
        )

        synthetic_df = pd.DataFrame(synthetic_data, columns=feature_columns)
        synthetic_df['Label'] = label
        synthetic_dfs.append(synthetic_df)

        print(f"  Label {label}: Generated {len(synthetic_df)} synthetic samples ✓")

    except Exception as e:
        print(f"  ⚠️ Label {label}: Error — {type(e).__name__}: {e}")
        print(f"  Falling back to noisy duplication.")
        duplicated = class_data.sample(n=samples_needed, replace=True, random_state=42)
        noise = np.random.normal(0, 0.01, duplicated.shape)
        duplicated = duplicated + noise
        duplicated['Label'] = label
        synthetic_dfs.append(duplicated)
        print(f"  Label {label}: Created {len(duplicated)} noisy duplicates ✓")

    # Free GPU memory after each class
    if USE_GPU:
        torch.cuda.empty_cache()

print(f"\n{'='*60}")
print("All WGAN-GP augmentation complete!")
print(f"{'='*60}")

In [ ]:
if synthetic_dfs:
    all_synthetic = pd.concat(synthetic_dfs, ignore_index=True)
    balanced_df = pd.concat([train_df, all_synthetic], ignore_index=True)
else:
    balanced_df = train_df.copy()

X_train_balanced = balanced_df.drop('Label', axis=1).values
y_train_balanced = balanced_df['Label'].values

print("WGAN-GP BALANCED TRAINING SET:")
print(pd.Series(y_train_balanced).value_counts())
print(f"\nTotal training samples: {len(y_train_balanced):,}")

## Step 7: Scale Features

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_balanced)
X_test_scaled = scaler.transform(X_test)

print(f"Features scaled successfully!")
print(f"Train mean: {X_train_scaled.mean():.4f}, std: {X_train_scaled.std():.4f}")
print(f"Test mean: {X_test_scaled.mean():.4f}, std: {X_test_scaled.std():.4f}")

## Step 8: Train SVM (SGDClassifier - Hinge Loss)

In [ ]:
print("Training SVM (SGDClassifier with Hinge Loss)...")
svm_model = SGDClassifier(
    loss='hinge',
    class_weight='balanced',
    random_state=42,
    max_iter=1000,
    tol=1e-3,
    n_jobs=-1
)
svm_model.fit(X_train_scaled, y_train_balanced)

y_train_pred_svm = svm_model.predict(X_train_scaled)
y_test_pred_svm = svm_model.predict(X_test_scaled)

svm_metrics = {
    'Algorithm': 'SVM (SGDClassifier - Hinge Loss)',
    'Train Accuracy': accuracy_score(y_train_balanced, y_train_pred_svm),
    'Test Accuracy': accuracy_score(y_test, y_test_pred_svm),
    'Precision': precision_score(y_test, y_test_pred_svm, average='weighted', zero_division=0),
    'Recall': recall_score(y_test, y_test_pred_svm, average='weighted', zero_division=0),
    'F1-Score': f1_score(y_test, y_test_pred_svm, average='weighted', zero_division=0)
}

print(f"SVM Training Complete!")
print(f"  Train Accuracy: {svm_metrics['Train Accuracy']:.4f}")
print(f"  Test Accuracy: {svm_metrics['Test Accuracy']:.4f}")
print(f"  Precision: {svm_metrics['Precision']:.4f}")
print(f"  Recall: {svm_metrics['Recall']:.4f}")
print(f"  F1-Score: {svm_metrics['F1-Score']:.4f}")

## Step 9: Train Random Forest

In [ ]:
print("Training Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced_subsample'
)
rf_model.fit(X_train_balanced, y_train_balanced)

y_train_pred_rf = rf_model.predict(X_train_balanced)
y_test_pred_rf = rf_model.predict(X_test)

rf_metrics = {
    'Algorithm': 'Random Forest',
    'Train Accuracy': accuracy_score(y_train_balanced, y_train_pred_rf),
    'Test Accuracy': accuracy_score(y_test, y_test_pred_rf),
    'Precision': precision_score(y_test, y_test_pred_rf, average='weighted', zero_division=0),
    'Recall': recall_score(y_test, y_test_pred_rf, average='weighted', zero_division=0),
    'F1-Score': f1_score(y_test, y_test_pred_rf, average='weighted', zero_division=0)
}

print(f"Random Forest Training Complete!")
print(f"  Train Accuracy: {rf_metrics['Train Accuracy']:.4f}")
print(f"  Test Accuracy: {rf_metrics['Test Accuracy']:.4f}")
print(f"  Precision: {rf_metrics['Precision']:.4f}")
print(f"  Recall: {rf_metrics['Recall']:.4f}")
print(f"  F1-Score: {rf_metrics['F1-Score']:.4f}")

## Step 10: Compare Results

In [ ]:
comparison_df = pd.DataFrame([svm_metrics, rf_metrics])
print("\n" + "="*80)
print("MODEL PERFORMANCE COMPARISON (WGAN-GP Balanced Dataset)")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

## Step 11: Visualize Metrics (Line Plot)

In [ ]:
metrics_data = comparison_df.set_index('Algorithm').T

fig, ax = plt.subplots(figsize=(12, 6))
metrics_data.plot(ax=ax, marker='o', linewidth=2, markersize=8)
ax.set_ylabel('Score', fontsize=12)
ax.set_xlabel('Metrics', fontsize=12)
ax.set_title('SVM vs Random Forest - Performance Metrics (WGAN-GP Balanced)', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)

for col in metrics_data.columns:
    for idx, val in enumerate(metrics_data[col]):
        ax.text(idx, val + 0.02, f'{val:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## Step 12: Visualize Metrics (Bar Plot)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

metrics_to_plot = ['Train Accuracy', 'Test Accuracy', 'Precision', 'Recall', 'F1-Score']
colors = ['#1f77b4', '#ff7f0e']

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx]
    values = [svm_metrics[metric], rf_metrics[metric]]
    bars = ax.bar(['SVM', 'Random Forest'], values, color=colors, alpha=0.8, edgecolor='black')
    ax.set_ylabel(metric, fontsize=11)
    ax.set_ylim([0, 1.05])
    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

fig.delaxes(axes[5])
fig.suptitle('SVM vs Random Forest - Detailed Metrics (WGAN-GP Balanced)', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## Step 13: Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_svm = confusion_matrix(y_test, y_test_pred_svm)
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=True)
axes[0].set_title('SVM Confusion Matrix', fontsize=12, fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

cm_rf = confusion_matrix(y_test, y_test_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Oranges', ax=axes[1], cbar=True)
axes[1].set_title('Random Forest Confusion Matrix', fontsize=12, fontweight='bold')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

## Step 14: Classification Reports

In [ ]:
print("\n" + "="*80)
print("SVM - CLASSIFICATION REPORT")
print("="*80)
print(classification_report(y_test, y_test_pred_svm, zero_division=0))

print("\n" + "="*80)
print("RANDOM FOREST - CLASSIFICATION REPORT")
print("="*80)
print(classification_report(y_test, y_test_pred_rf, zero_division=0))

## Step 15: Summary

In [ ]:
print("\n" + "#"*80)
print("# SUMMARY")
print("#"*80)

best_f1 = 'SVM' if svm_metrics['F1-Score'] > rf_metrics['F1-Score'] else 'Random Forest'
best_acc = 'SVM' if svm_metrics['Test Accuracy'] > rf_metrics['Test Accuracy'] else 'Random Forest'

print(f"\nDataset: CICIDS2017 (Cleaned + WGAN-GP Balanced)")
print(f"Total Samples (Original): {len(df):,}")
print(f"Total Training Samples (After WGAN-GP): {len(y_train_balanced):,}")
print(f"Total Features: {X.shape[1]}")
print(f"Train/Test Split: 70/30")
print(f"Test Samples: {len(X_test):,}")
print(f"\nBalancing Strategy:")
print(f"  - Classes > {threshold}: Undersampled to 50,000")
print(f"  - Classes <= {threshold}: WGAN-GP augmented to {target_minority}")

print(f"\n" + "-"*80)
print(f"BEST MODEL (by Test Accuracy): {best_acc}")
print(f"BEST MODEL (by F1-Score): {best_f1}")
print(f"-"*80)

print(f"\nSVM Performance:")
for key, value in svm_metrics.items():
    if key != 'Algorithm':
        print(f"  {key}: {value:.4f}")

print(f"\nRandom Forest Performance:")
for key, value in rf_metrics.items():
    if key != 'Algorithm':
        print(f"  {key}: {value:.4f}")

print(f"\n" + "#"*80)

## Step 16: Save Models to Drive

In [ ]:
import joblib
import os

save_path = '/content/drive/MyDrive/IDS_Project/saved_models_wgan_gp'
os.makedirs(save_path, exist_ok=True)

joblib.dump(svm_model, f'{save_path}/svm_model_wgan_gp.pkl')
joblib.dump(rf_model, f'{save_path}/rf_model_wgan_gp.pkl')
joblib.dump(scaler, f'{save_path}/scaler_wgan_gp.pkl')

print("All models saved to Google Drive!")
print(f"Location: {save_path}")